# Phase 2+ M1: セグメンテーション学習 (YOLOv8n-seg)

## 目的

本プロジェクトは Phase 3 で **物体検出 mAP@0.5 = 0.966** を達成した。しかし「3D で家として見られる Viewer」を実現するには、設備記号の位置情報だけでなく、**部屋の輪郭(room)** と **壁(wall)** のセグメンテーション情報が必要。

本ノートブックでは、物体検出と並行して、Phase 1+ で復活させた room/wall を含む 8 クラスで YOLOv8n-seg を学習する。

## 仮説

**Exp 2 で確立された最良ハイパラ(imgsz=1024, epochs=100)で YOLOv8n-seg を学習すれば、room と wall を十分な精度でセグメンテーションでき、後段の 3D 化処理に使えるレベルになる。**

予想:
- door / window / toilet / sink などは Box mAP が物体検出版(>0.95)と同等
- **room の Mask mAP > 0.7** であれば 3D 化に使える
- **wall の Mask mAP > 0.5** であれば壁線抽出に使える
- shower / staircase は引き続き難易度高め

## 設定(Exp 2 と公正な比較のため、解像度・epochs を踏襲)

| 項目 | Exp 2 (検出) | M1 (セグメンテーション) | 変更 |
|---|---|---|---|
| **Task** | detect | **segment** | ⭐ |
| **Model** | yolov8n.pt | **yolov8n-seg.pt** | ⭐ |
| **Classes** | 6 | **8** (room/wall 復活) | ⭐ |
| imgsz | 1024 | 1024 | 同じ |
| epochs | 100 | 100 | 同じ |
| batch | 8 | 4 | セグでメモリ多消費のため減 |
| seed | 42 | 42 | 同じ |
| patience | 25 | 25 | 同じ |

## Section 1: 環境セットアップ

In [ ]:
!nvidia-smi

In [ ]:
import os

WORKDIR = "/content/floor-plan-recognition"

if not os.path.exists(WORKDIR):
    !git clone https://github.com/Mao925/floor-plan-recognition.git {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull

%cd {WORKDIR}
!pwd

In [ ]:
!pip install -q ultralytics roboflow python-dotenv

In [ ]:
import torch
import ultralytics

print(f"PyTorch:        {torch.__version__}")
print(f"Ultralytics:    {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Section 2: データ準備

In [ ]:
from google.colab import userdata

try:
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print(f"✅ API キー取得成功: {ROBOFLOW_API_KEY[:3]}***{ROBOFLOW_API_KEY[-3:]}")
except Exception as e:
    print(f"❌ エラー: {e}")

In [ ]:
with open('.env', 'w') as f:
    f.write(f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')

# 元データを取得(Phase 1 と同じ)
!python scripts/download_roboflow.py

In [ ]:
# セグメンテーション用に前処理(8クラス、polygon維持)
!python scripts/prepare_dataset_seg.py

In [ ]:
!cat data/floorplan_seg/data.yaml

## Section 3: 学習(YOLOv8n-seg / imgsz=1024 / epochs=100)

**所要時間予想**: T4 GPU で約 30〜60 分(セグメンテーションは検出より重い)

**注意**: 
- セグはメモリを多く使うため batch=4 に設定
- OOM が出たら batch を 2 に下げる

In [ ]:
from ultralytics import YOLO

# セグメンテーション版の事前学習モデル
model = YOLO('yolov8n-seg.pt')

results = model.train(
    data='data/floorplan_seg/data.yaml',
    task='segment',
    epochs=100,
    imgsz=1024,
    batch=4,                  # セグはメモリ多消費のため減
    name='m1_seg_yolov8n',
    project='runs/segment',
    patience=25,
    save=True,
    plots=True,
    device=0,
    seed=42,
)

print("\n✅ セグメンテーション学習完了")

In [ ]:
# 学習結果のパスを確認
!find runs -name 'best.pt' -path '*m1_seg*' | head -5

## Section 4: 評価

セグメンテーションでは **Box mAP**(物体検出と同じ指標)と **Mask mAP**(セグメンテーション専用指標)の両方が見られる。

In [ ]:
from pathlib import Path

candidates = list(Path('runs').rglob('m1_seg_yolov8n/weights/best.pt'))
assert candidates, "best.pt が見つかりません"
best_pt = candidates[0]
results_dir = best_pt.parent.parent
print(f"学習結果フォルダ: {results_dir}")
print(f"ベストモデル:     {best_pt}")

In [ ]:
from IPython.display import Image, display

for img_name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    img_path = results_dir / img_name
    if img_path.exists():
        print(f"\n=== {img_name} ===")
        display(Image(str(img_path)))

In [ ]:
# test セット最終評価
best_model = YOLO(str(best_pt))

test_metrics = best_model.val(
    data='data/floorplan_seg/data.yaml',
    split='test',
    imgsz=1024,
    name='m1_seg_test_eval',
    project='runs/segment',
)

print("\n=== Test セット全体メトリクス (M1: セグメンテーション) ===")
print(f"\n[Box (物体検出 bbox の精度)]")
print(f"  mAP@0.5:        {test_metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95:   {test_metrics.box.map:.4f}")
print(f"\n[Mask (セグメンテーション の精度)]")
print(f"  mAP@0.5:        {test_metrics.seg.map50:.4f}")
print(f"  mAP@0.5:0.95:   {test_metrics.seg.map:.4f}")

In [ ]:
# クラス別 Box と Mask の AP
import pandas as pd

class_names = ['door', 'room', 'shower', 'sink', 'staircase', 'toilet', 'wall', 'window']

box_aps = test_metrics.box.ap50
mask_aps = test_metrics.seg.ap50

df = pd.DataFrame({
    'Box AP@0.5':  [float(box_aps[i])  for i in range(len(class_names))],
    'Mask AP@0.5': [float(mask_aps[i]) for i in range(len(class_names))],
}, index=class_names)
df['Box vs Mask'] = df['Box AP@0.5'] - df['Mask AP@0.5']

print("=" * 70)
print("クラス別 AP@0.5: Box vs Mask")
print("=" * 70)
print(df.to_string(float_format=lambda x: f'{x:.4f}'))

print("\n=== 3D 化に向けた重要クラスの評価 ===")
for k in ['room', 'wall']:
    print(f"  {k} Mask AP@0.5: {df.loc[k, 'Mask AP@0.5']:.4f}")
    if df.loc[k, 'Mask AP@0.5'] > 0.5:
        print(f"    → 3D 化に使えるレベル ✅")
    else:
        print(f"    → 改善が必要 ⚠️")

In [ ]:
# 推論サンプル(セグメンテーションで部屋・壁が見える)
import random

test_images = sorted(Path('data/floorplan_seg/test/images').glob('*.jpg'))
random.seed(42)
samples = random.sample(test_images, min(4, len(test_images)))

predict_results = best_model.predict(
    source=[str(p) for p in samples],
    imgsz=1024,
    save=True,
    project='runs/segment',
    name='m1_seg_samples',
    conf=0.25,
)

pred_dir = list(Path('runs').rglob('m1_seg_samples'))[0]
for img_path in sorted(pred_dir.glob('*.jpg')):
    print(f"\n=== {img_path.name} ===")
    display(Image(str(img_path)))

## Section 5: 結果の保存

In [ ]:
import shutil

src = str(results_dir)
out_zip = '/content/m1_seg_results.zip'
shutil.make_archive(out_zip.replace('.zip', ''), 'zip', src)
print(f"✅ Zip 作成完了: {out_zip}")
!ls -lh {out_zip}

In [ ]:
from google.colab import files
files.download(out_zip)

---

## 検証結果のまとめ(実験完了後にここに記入する)

### M1 達成条件
- [ ] 学習が完走した(または有意義な epoch で EarlyStopping)
- [ ] **room Mask AP@0.5 > 0.5**(3D 化に使える最低ライン)
- [ ] **wall Mask AP@0.5 > 0.4**(壁線抽出に使える最低ライン)
- [ ] door/window などの主要クラスは Box AP@0.5 > 0.9 を維持

### 結果
- 全体 Box mAP@0.5: ?
- 全体 Mask mAP@0.5: ?
- room Mask AP@0.5: ?
- wall Mask AP@0.5: ?

### 解釈と次のステップ
- room と wall が使えるレベルなら → Phase 4(ベクトル化)へ進める
- 使えないレベルなら → epochs 増 or アーキテクチャ変更を検討